# Fine-Tune Whisper For Multi-Accented English Speech with 🤗 Transformers - Speech Technology

In this Colab, we present a step-by-step guide on how to fine-tune Whisper
for any multilingual ASR dataset using Hugging Face 🤗 Transformers. This notebook is adapted from [Whisper Finetuning Event ](https://huggingface.co/blog/fine-tune-whisper)and modified for this assignment.

When you select your GPU, the credits usage might differ depending on the GPU selected and runtime. Here are some important considerations:

* **GPU Selection and Batch Size:**

 * A100 GPU: If you are using an A100, which has 40 GB of memory, you can increase the batch_size along with gradient_accumulation_steps in hyperparameters for faster training. This GPU is suitable for handling larger batch sizes and more complex models.

 * T4 GPU: If you use a T4 GPU, which has 15 GB of memory, it is recommended to use a smaller batch_size with gradient_accumulation_steps modified accordingly. This will help avoid out-of-memory errors and ensure stable training.

* **Read the Notebook First:** It is recommended to read through this notebook before starting any experiments. Understanding the workflow and the purpose of each section will give you a clear idea of what to expect and how to proceed.

* **Manage Runtime Usage:** Make sure to close your Runtime when your experiments are finished. Leaving the runtime open will continue to consume GPU memory in the background, even if you are not actively using it. This can lead to unnecessary credit usage and potentially limit availability for other using Colab.

By following these instructions, you can optimize your usage of Colab resources and ensure a smooth experience while working on this assignment.

## Introduction

Whisper is a pre-trained model for automatic speech recognition (ASR)
published in [September 2022](https://openai.com/blog/whisper/) by the authors
Alec Radford et al. from OpenAI. Unlike many of its predecessors, such as
[Wav2Vec 2.0](https://arxiv.org/abs/2006.11477), which are pre-trained
on un-labelled audio data, Whisper is pre-trained on a vast quantity of
**labelled** audio-transcription data, 680,000 hours to be precise.
This is an order of magnitude more data than the un-labelled audio data used
to train Wav2Vec 2.0 (60,000 hours). What is more, 117,000 hours of this
pre-training data is multilingual ASR data. This results in checkpoints
that can be applied to over 96 languages, many of which are considered
_low-resource_.

When scaled to 680,000 hours of labelled pre-training data, Whisper models
demonstrate a strong ability to generalise to many datasets and domains.
The pre-trained checkpoints achieve competitive results to state-of-the-art
ASR systems, with near 3% word error rate (WER) on the test-clean subset of
LibriSpeech ASR and a new state-of-the-art on TED-LIUM with 4.7% WER (_c.f._
Table 8 of the [Whisper paper](https://cdn.openai.com/papers/whisper.pdf)).
The extensive multilingual ASR knowledge acquired by Whisper during pre-training
can be leveraged for other low-resource languages; through fine-tuning, the
pre-trained checkpoints can be adapted for specific datasets and languages
to further improve upon these results. We'll show just how Whisper can be fine-tuned for low-resource English accented speech in this Colab.

<figure>
<img src="https://raw.githubusercontent.com/sanchit-gandhi/notebooks/main/whisper_architecture.svg" alt="Trulli" style="width:100%">
<figcaption align = "center"><b>Figure 1:</b> Whisper model. The architecture
follows the standard Transformer-based encoder-decoder model. A
log-Mel spectrogram is input to the encoder. The last encoder
hidden states are input to the decoder via cross-attention mechanisms. The
decoder autoregressively predicts text tokens, jointly conditional on the
encoder hidden states and previously predicted tokens. Figure source:
<a href="https://openai.com/blog/whisper/">OpenAI Whisper Blog</a>.</figcaption>
</figure>

The Whisper checkpoints come in five configurations of varying model sizes.
The smallest four are trained on either English-only or multilingual data.
The largest checkpoint is multilingual only. All nine of the pre-trained checkpoints
are available on the [Hugging Face Hub](https://huggingface.co/models?search=openai/whisper). The
checkpoints are summarised in the following table with links to the models on the Hub:

| Size   | Layers | Width | Heads | Parameters | English-only                                         | Multilingual                                      |
|--------|--------|-------|-------|------------|------------------------------------------------------|---------------------------------------------------|
| tiny   | 4      | 384   | 6     | 39 M       | [✓](https://huggingface.co/openai/whisper-tiny.en)   | [✓](https://huggingface.co/openai/whisper-tiny.)  |
| base   | 6      | 512   | 8     | 74 M       | [✓](https://huggingface.co/openai/whisper-base.en)   | [✓](https://huggingface.co/openai/whisper-base)   |
| small  | 12     | 768   | 12    | 244 M      | [✓](https://huggingface.co/openai/whisper-small.en)  | [✓](https://huggingface.co/openai/whisper-small)  |
| medium | 24     | 1024  | 16    | 769 M      | [✓](https://huggingface.co/openai/whisper-medium.en) | [✓](https://huggingface.co/openai/whisper-medium) |
| large  | 32     | 1280  | 20    | 1550 M     | x                                                    | [✓](https://huggingface.co/openai/whisper-large)  |

For demonstration purposes, we'll fine-tune the **multilingual** version of the
[`"tiny"`](https://huggingface.co/openai/whisper-tiny) checkpoint with 39M params.
As for our data, we'll train and evaluate our system on an unseen portion of the multi-accented dataset which is not used during training. This portion of the dataset contains different speakers than those used for training. We'll show that with as little as 2-3 hours of fine-tuning data, we can achieve
strong performance in this accent.

------------------------------------------------------------------------

\\({}^1\\) The name Whisper follows from the acronym “WSPSR”, which stands for “Web-scale Supervised Pre-training for Speech Recognition”.

## Prepare Environment

**Help ->** This is the part where you can see the GPU assigned to you on your Google Colab notebook. It's good to see beforehand the GPU memory allocated based on the Whisper finetuning experiments to be performed. Since we are finetuning on a Tiny Whisper model, anything with 16 GB of memory should suffice. Let's try to secure a decent GPU for our Colab! Even with the free version of Google Colab, you can get access to something like a Tesla T4 GPU which has 16 GB of memory, which is more than enough to run your experiment.

To get a GPU, click Runtime -> Change runtime type, then change Hardware accelerator from None to GPU.

We can verify that we've been assigned a GPU and view its specifications:

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

We'll employ several popular Python packages to fine-tune the Whisper model.
We'll use `datasets` to download and prepare our training data and
`transformers` to load and train our Whisper model. We'll also require
the `soundfile` package to pre-process audio files, `evaluate` and `jiwer` to
assess the performance of our model. Finally, `autotime` library to this which reports time each notebook cell takes to run. This gives a good idea of how much training time its taking to run the whisper finetuning.

In [ ]:
!pip install datasets>=2.6.1
!pip install git+https://github.com/huggingface/transformers
!pip install librosa
!pip install evaluate>=0.30
!pip install jiwer
!pip install ipython-autotime
!pip install accelerate -U
%load_ext autotime
!pip install -U datasets[audio]
# !pip install torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 torchcodec==0.9 => Read the section Quick Tips and Notes

**Help**-> You will first need to create a Hugging face account to create your own access tokens. User Access Tokens are the preferred way to authenticate an application or notebook to Hugging Face services. You can use the this [link](https://huggingface.co/docs/hub/en/security-tokens) to learn how to create your own access tokens. Note that access tokens are private API key to your Hugging face account, so it's advised not to share them with others.

We strongly advise you to upload model checkpoints directly to the [Hugging Face Hub](https://huggingface.co/)
whilst training. The Hub provides:
- Integrated version control: you can be sure that no model checkpoint is lost during training.
- Tensorboard logs: track important metrics over the course of training.
- Model cards: document what a model does and its intended use cases.
- Community: an easy way to share and collaborate with the community!

Linking the notebook to the Hub is straightforward - Just run the cell below, open the URL and enter the shown code when prompted. You do not need to grant permissions to any of your organisation for this notebook.





In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:

import torch
import transformers
import datasets
import evaluate
import accelerate
from importlib.metadata import version

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("Accelerate:", accelerate.__version__)
print("Jiwer:", version("jiwer"))

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


After obtaining your token, you can authenticate with the huggingface_hub by logging in with your token. This grants this notebook access to your Hugging Face account, allowing the trained model to be saved directly to your account. Simply run the cell above and paste your token when prompted.

## Load Dataset

Using 🤗 Datasets, downloading and preparing data is extremely simple.
We can download and prepare the assignment dataset splits with just one line of code. Below is an example of loading the Spanish English Dataset, but you can change the path according to your use case. We are using the common_voice data dictionary to load our dataset, which expects the data file in a specific format. Since we have already prepared the dataset in that format for this assignment, you can directly load it from the Hugging Face directory.


In [ ]:
from datasets import load_dataset, DatasetDict

common_voice = DatasetDict()

common_voice["train"] = load_dataset("naharte/korean_english", split="train")
common_voice["test"] = load_dataset("naharte/korean_english", split="test")


In [ ]:
common_voice

In [ ]:
common_voice = common_voice.rename_column("transcription", "sentence") # This step just change the "transcription" column name to "sentence" for standardisation.
print(common_voice)

Most ASR datasets only provide input audio samples (`audio`) and the
corresponding transcribed text (`sentence`). So we have prepared our dataset in a similar format. Since "sentence" is used as variable for doing preprocessing in the later stages of this colab. We will change the "transcription" column to "sentence" as shown above.

## Prepare Feature Extractor, Tokenizer and Data

The ASR pipeline can be de-composed into three stages:
1) A feature extractor which pre-processes the raw audio-inputs
2) The model which performs the sequence-to-sequence mapping
3) A tokenizer which post-processes the model outputs to text format

In 🤗 Transformers, the Whisper model has an associated feature extractor and tokenizer,
called [WhisperFeatureExtractor](https://huggingface.co/docs/transformers/main/model_doc/whisper#transformers.WhisperFeatureExtractor)
and [WhisperTokenizer](https://huggingface.co/docs/transformers/main/model_doc/whisper#transformers.WhisperTokenizer)
respectively.

We'll go through details for setting-up the feature extractor and tokenizer one-by-one!

### Load WhisperFeatureExtractor

The Whisper feature extractor performs two operations:
1. Pads / truncates the audio inputs to 30s: any audio inputs shorter than 30s are padded to 30s with silence (zeros), and those longer that 30s are truncated to 30s
2. Converts the audio inputs to _log-Mel spectrogram_ input features, a visual representation of the audio and the form of the input expected by the Whisper model

<figure>
<img src="https://raw.githubusercontent.com/sanchit-gandhi/notebooks/main/spectrogram.jpg" alt="Trulli" style="width:100%">
<figcaption align = "center"><b>Figure 2:</b> Conversion of sampled audio array to log-Mel spectrogram.
Left: sampled 1-dimensional audio signal. Right: corresponding log-Mel spectrogram. Figure source:
<a href="https://ai.googleblog.com/2019/04/specaugment-new-data-augmentation.html">Google SpecAugment Blog</a>.
</figcaption>

We'll load the feature extractor from the pre-trained checkpoint with the default values:

In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")

### Load WhisperTokenizer

The Whisper model outputs a sequence of _token ids_. The tokenizer maps each of these token ids to their corresponding text string. The tokenizer assume language to be "English" by default.

In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", task="transcribe")

### Combine To Create A WhisperProcessor

To simplify using the feature extractor and tokenizer, we can _wrap_
both into a single `WhisperProcessor` class. This processor object
inherits from the `WhisperFeatureExtractor` and `WhisperProcessor`,
and can be used on the audio inputs and model predictions as required.
In doing so, we only need to keep track of two objects during training:
the `processor` and the `model`:

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny",  task="transcribe")

### Prepare Data

Let's print the first example of the Accented dataset to see
what form the data is in:

In [ ]:
index = 0
print(common_voice["train"][index])
from IPython.display import Audio

samples = common_voice["train"][index]["audio"].get_all_samples()

In [ ]:
Audio(samples.data.numpy(), rate=samples.sample_rate) # Print the corresponding audio

Our input audio is sampled at 16kHz, which is the required rate for this finetuning. However, sometimes the audio may be presented at different sampling rates. To ensure compatibility, we will change the sampling rate of all audio files to 16kHz before passing them to the Whisper feature extractor, as 16kHz is the expected sampling rate for the Whisper model.

We'll set the audio inputs to the correct sampling rate using dataset's
[`cast_column`](https://huggingface.co/docs/datasets/package_reference/main_classes.html?highlight=cast_column#datasets.DatasetDict.cast_column)
method. This operation does not change the audio in-place,
but rather signals to `datasets` to resample audio samples _on the fly_ the
first time that they are loaded:

In [ ]:
from datasets import Audio

common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

Re-loading the first audio sample in the Common Voice dataset will resample
it to the desired sampling rate:

In [ ]:
common_voice["train"][0]

Now we can write a function to prepare our data ready for the model:
1. We load and resample the audio data by calling `batch["audio"]`. As explained above, 🤗 Datasets performs any necessary resampling operations on the fly.
2. We use the feature extractor to compute the log-Mel spectrogram input features from our 1-dimensional audio array.
3. We encode the transcriptions to label ids through the use of the tokenizer.

In [ ]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

We can apply the data preparation function to all of our training examples using dataset's `.map` method. The argument `num_proc` specifies how many CPU cores to use. Setting `num_proc` > 1 will enable multiprocessing. If the `.map` method hangs with multiprocessing, set `num_proc=1` and process the dataset sequentially.

In [ ]:
common_voice = common_voice.map(prepare_dataset, remove_columns=common_voice.column_names["train"], num_proc=2) # here we remove the old columns to save the space complexity.
# Can increase num_proc to reduce the running time.

In [ ]:
common_voice

## Training and Evaluation

Now that we've prepared our data, we're ready to dive into the training pipeline.
The [🤗 Trainer](https://huggingface.co/transformers/master/main_classes/trainer.html?highlight=trainer)
will do much of the heavy lifting for us. All we have to do is:

- Define a data collator: the data collator takes our pre-processed data and prepares PyTorch tensors ready for the model.

- Evaluation metrics: during evaluation, we want to evaluate the model using the [word error rate (WER)](https://huggingface.co/metrics/wer) metric. We need to define a `compute_metrics` function that handles this computation.

- Load a pre-trained checkpoint: we need to load a pre-trained checkpoint and configure it correctly for training.

- Define the training configuration: this will be used by the 🤗 Trainer to define the training schedule.

Once we've fine-tuned the model, we will evaluate it on the test data to verify that we have correctly trained it
to transcribe speech in Spanish accent.

### Define a Data Collator

The data collator for a sequence-to-sequence speech model is unique in the sense that it
treats the `input_features` and `labels` independently: the  `input_features` must be
handled by the feature extractor and the `labels` by the tokenizer.

The `input_features` are already padded to 30s and converted to a log-Mel spectrogram
of fixed dimension by action of the feature extractor, so all we have to do is convert the `input_features`
to batched PyTorch tensors. We do this using the feature extractor's `.pad` method with `return_tensors=pt`.

The `labels` on the other hand are un-padded. We first pad the sequences
to the maximum length in the batch using the tokenizer's `.pad` method. The padding tokens
are then replaced by `-100` so that these tokens are **not** taken into account when
computing the loss. We then cut the BOS token from the start of the label sequence as we
append it later during training.

We can leverage the `WhisperProcessor` we defined earlier to perform both the
feature extractor and the tokenizer operations:

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

Let's initialise the data collator we've just defined:

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

### Evaluation Metrics

We'll use the word error rate (WER) metric, the 'de-facto' metric for assessing
ASR systems. For more information, refer to the WER [docs](https://huggingface.co/metrics/wer). We'll load the WER metric from 🤗 Evaluate:

In [ ]:
import evaluate
metric = evaluate.load("wer")

We then simply have to define a function that takes our model
predictions and returns the WER metric. This function, called
`compute_metrics`, first replaces `-100` with the `pad_token_id`
in the `label_ids` (undoing the step we applied in the
data collator to ignore padded tokens correctly in the loss).
It then decodes the predicted and label ids to strings. Finally,
it computes the WER between the predictions and reference labels:

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

### Load a Pre-Trained Checkpoint

Now let's load the pre-trained Whisper `tiny` checkpoint. Again, this
is trivial through use of 🤗 Transformers!

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

Override generation arguments - no tokens are forced as decoder outputs (see [`forced_decoder_ids`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate.forced_decoder_ids)), no tokens are suppressed during generation (see [`suppress_tokens`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate.suppress_tokens)):

In [ ]:
# This block is used to resolve the transformers conflicts between versions. Whisper was trained with an old transformer version, which might have staled arguments the its config.
gen_params = model.config._get_generation_parameters()
for key, value in gen_params.items():
    setattr(model.generation_config, key, value)  # keep the setting where generate() expects it
    setattr(model.config, key, None)              # clear it from the model config

### Define the Training Configuration

In the final step, we define all the parameters related to training. For more detail on the training arguments, refer to the Seq2SeqTrainingArguments [docs](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainingArguments).

**Note:** Ensure that you change the name of the output directory to prevent it from overwriting any previous directories if you're using the same notebook. This section also allows you to modify hyperparameters for finetuning. While this notebook uses the default parameters, you can experiment with different parameters (such as batch_size, learning_rate, max_steps, etc.) to achieve optimal training results.

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny_to_korean_accent_29oct",  # change to a repo name of your choice
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=2000,
    gradient_checkpointing=True,
    eval_strategy="steps",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)

**Note**: if one does not want to upload the model checkpoints to the Hub,
set `push_to_hub=False`. We recommend having this option set to True. However, if you are just playing with this model, and doesn't want anything to be uploaded to your HuggingFace hub, you can turn it off.

We can forward the training arguments to the 🤗 Trainer along with our model,
dataset, data collator and `compute_metrics` function:

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
# If encounter 403 Forbidden: You don't have the rights to create a model under the namespace -> make sure your token has WRITE access

We'll save the processor object once before starting training. Since the processor is not trainable, it won't change over the course of training:

In [ ]:
processor.save_pretrained(training_args.output_dir)

### Training

Training will take approximately 10 minutes for 2,000 steps depending on your GPU or the one
allocated to this Google Colab.

Depending on the GPU allocated to the Google Colab, it is possible that you will encounter a CUDA `"out-of-memory"` error when you launch training.
In this case, you can reduce the `per_device_train_batch_size` incrementally by factors of 2
and employ [`gradient_accumulation_steps`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainingArguments.gradient_accumulation_steps)
to compensate.

To launch training, simply execute:

In [ ]:
trainer.train()

You will receive different WER for different multiaccented dataset. You can use your finetuned model and compare the performance with other non-finetuned models using the Whisper-Inference notebook.

In [ ]:

print("Current global step:", trainer.state.global_step)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best evaluation metric:", trainer.state.best_metric)

print("Model class:", type(trainer.model).__name__)
print("Output projection shape:", trainer.model.proj_out.weight.shape)
print("All projection weights finite:",
      bool(torch.isfinite(trainer.model.proj_out.weight).all()))


In [ ]:
kwargs = {
    "dataset_tags": "Korean_english",  # Example valid dataset ID from Hugging Face
    "dataset": "Korean English",       # Pretty name for the training dataset
    "dataset_args": "config: default, split: test",
    "language": "en",                # ISO 639-1 code for English
    "model_name": "Whisper tiny Korean",
    "finetuned_from": "openai/whisper-tiny",
    "tasks": "automatic-speech-recognition",
    "tags": "hf-asr-leaderboard",
}


The training results can now be uploaded to the Hub. To do so, execute the `push_to_hub` command and save the preprocessor object we created:

In [ ]:
trainer.push_to_hub(**kwargs)

### Evaluation

In [ ]:
trainer.evaluate() # it reads the dataset and collator definition from trainer (cell 36 above)

## Analysis
We can analyse the transcriptions with the help of `jiwer` library. Remember that we defined our own WER calculator above (`compute_metrics` function, using the `evaluate` library). `jiwer` should have the same computation logic, but it offers more handy functions to visualise the alignment between the predicted and ground-truth transcriptions.

In [ ]:
output = trainer.predict(common_voice["test"])

pred_ids = output.predictions
label_ids = output.label_ids

# -100 marks padding/ignored positions; swap it for the pad token before decoding
pad_id = processor.tokenizer.pad_token_id
pred_ids[pred_ids == -100] = pad_id
label_ids[label_ids == -100] = pad_id

hypotheses = processor.batch_decode(pred_ids, skip_special_tokens=True)
references = processor.batch_decode(label_ids, skip_special_tokens=True)

for ref, hyp in list(zip(references, hypotheses))[:5]:
    print(f"REF: {ref}\nHYP: {hyp}\n")


In [ ]:
from jiwer import wer
error = wer(references, hypotheses)
print(error) # should be the same (or at least, roughly the same!) value with `trainer.evaluate()` output.

We can save the predictions and its corresponding reference in a csv file to analyse later. Note that you **HAVE TO** download the csv files to your locally manually.   
Since we haven't mounted Google Drive from this notebook, all files are just temporarily saved on the cloud and will disappear after this notebook is reset.

In [ ]:
import pandas as pd
df = pd.DataFrame({"reference": references, "hypothesis": hypotheses})
df.to_csv(f"{training_args.output_dir}/whisper_preds.csv", index=False) # SAVE THIS!!

Now we can use the `visualize_alignment` function from `jiwer` to compare a hypothesis and a reference transcription.  

In [ ]:
'''
Example:
REF: dont solve the brush with paint just put a little on
HYP: dont  soak the brush with paint just put a little on
              S

=> "solve" is substituted (S) with "soak" in the hypothesis.
'''

In [ ]:
import jiwer

out = jiwer.process_words(
    references[:3],
    hypotheses[:3]
)
print(jiwer.visualize_alignment(out))

You can find more usages of `jiwer` here: https://jitsi.github.io/jiwer/usage.  
You may try analysing different kinds of errors (substituion, insertion, deletion) on each accent.

In this notebook, we covered a step-by-step guide on fine-tuning Whisper for multiaccented English ASR
using 🤗 Datasets, Transformers and the Hugging Face Hub. For more details on the Whisper model and the theory behind fine-tuning, refere to the accompanying [blog post](https://huggingface.co/blog/fine-tune-whisper). If you're interested in fine-tuning other
Transformers models, both for English and multilingual ASR, be sure to check out the
examples scripts at [examples/pytorch/speech-recognition](https://github.com/huggingface/transformers/tree/main/examples/pytorch/speech-recognition).

## Quick Tips and Notes

If you are not familiar with PyTorch and/or Torchaudio, Torchcodec, etc. Here is the rule of thumb for debugging if you encounter any problem related to Torchaudio/Torch in this notebook: Make sure you install the correct and compatible versions of Torch, Torchaudio and Torchcodec.
Refer to this version compatibility in the Table [here](https://github.com/meta-pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec).

All data uploaded to HuggingFace are stored in thr parquet format by default (read more [here](https://huggingface.co/blog/parquet-cdc)). In our dataset, there are two features (columns): `audio` and `transcription`, in which `audio` is an array itself. If you wish to download the playable audio files for further analyses, try the code below.

In [ ]:
from datasets import load_dataset
import os
import soundfile as sf
from tqdm import tqdm

# Load dataset
dataset = load_dataset("naharte/korean_english", split="test", streaming=True) # replace dataset name and split here, keep streaming=True

# Paths
output_dir = "/content/audio_export" # specify your output here
os.makedirs(output_dir, exist_ok=True)

# the two below are standard files in kaldi recipes, which you may encounter a lot in the future if you work in speech technology and speech models training.
utt2dur_path = os.path.join(output_dir, "utt2dur") # this is a file of the format <path><duration in seconds>
scp_text_lang_path = os.path.join(output_dir, "wav.scp") # this is a file of the format <path><transcription>

# Open files for writing
with open(utt2dur_path, "w", encoding="utf-8") as f_dur, open(scp_text_lang_path, "w", encoding="utf-8") as f_text:
    for i, example in enumerate(tqdm(dataset, desc="Processing")):
        audio = example['audio'] # make sure this column exists in HF dataset card
        text = example['transcription'] # make sure this column exists in HF dataset card
        audio_name = f"audio_{i:04d}" # audio names are not defined in huggingface -> need to rename
        out_path = os.path.join(output_dir, f"{audio_name}.wav")

        # Save audio
        sf.write(out_path, audio['array'], audio['sampling_rate'])

        # Write to utt2dur
        duration = len(audio['array']) / audio['sampling_rate']
        f_dur.write(f"{audio_name}\t{duration:.2f}\n")

        # Write to wav.scp
        f_text.write(f"{out_path}\t{text}\n")

In [ ]:
import os

print("Drive mounted:", os.path.ismount("/content/drive"))

print("Contents of /content/drive:")
print(os.listdir("/content/drive"))

In [ ]:
from google.colab import drive
from pathlib import Path

mount_path = "/content/google_drive"
Path(mount_path).mkdir(exist_ok=True)

drive.mount(mount_path)

print("Drive mounted successfully")

In [ ]:

import pandas as pd
from pathlib import Path
from jiwer import wer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# ==================================================
# CONFIGURATION — Change only this for each accent
# ==================================================

accent = "korean"  # canadian, indian, chinese, japanese, korean

expected_counts = {
    "canadian": 407,
    "indian": 650,
    "chinese": 629,
    "japanese": 700,
    "korean": 450
}

# ==================================================
# Calculate WER
# ==================================================

normalizer = BasicTextNormalizer()

normalized_refs = [normalizer(text) for text in references]
normalized_hyps = [normalizer(text) for text in hypotheses]

raw_wer = wer(references, hypotheses)
normalized_wer = wer(normalized_refs, normalized_hyps)

print(f"Accent: {accent.capitalize()}")
print(f"Fine-tuned Raw WER: {raw_wer * 100:.2f}%")
print(f"Fine-tuned Normalized WER: {normalized_wer * 100:.2f}%")

# ==================================================
# Verify dataset size
# ==================================================

assert accent in expected_counts, f"Unknown accent: {accent}"

assert len(references) == len(hypotheses), "Reference/prediction count mismatch"

assert len(references) == expected_counts[accent], (
    f"Expected {expected_counts[accent]} recordings, "
    f"got {len(references)}"
)

# ==================================================
# Save predictions to Google Drive
# ==================================================

results_dir = Path(
    "/content/assignment_results"
)
results_dir.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame({
    "reference": references,
    "hypothesis": hypotheses,
    "normalized_reference": normalized_refs,
    "normalized_hypothesis": normalized_hyps
})

output_path = results_dir / f"{accent}_finetuned_predictions.csv"

df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Number of recordings: {len(df)}")
